Reproduction of "bad" container timing result

In [1]:
from egglog.exp.param_eq.original_results import load_original_results

rows = load_original_results()
bad_row = rows[rows["orig_parsed_expr"].str.startswith("-0.0314")]
bad_row

,dataset,raw_index,algorithm_raw,algorithm,algo_row,input_kind,n_params,n_rank,before_nodes,before_params,...,orig_expr,simpl_expr,orig_parsed_expr,simpl_parsed_expr,orig_parsed_n_params,simpl_parsed_n_params,before_rank_difference,after_rank_difference,before_parsed_rank_difference,after_parsed_rank_difference
6,kotanchek,4,Bingo,Bingo,4,original,7.0,5.0,28.0,8.0,...,(-0.03144312866911644)*(-3.0969157782045578 + ...,(-3.144312866911644e-2) * (((-3.09691577820455...,-0.03144312866911644 * (-3.0969157782045578 + ...,-0.03144312866911644 * (-3.0969157782045578 + ...,4.0,4.0,3.0,1.0,-1.0,-1.0


In [ ]:
from egglog.exp.param_eq.domain import binary_to_containers
from egglog.exp.param_eq.pipeline import *
from egglog import *


bad_expr = parse_expression(bad_row["orig_parsed_expr"].iloc[0])
bad_expr_container = binary_to_containers(bad_expr)

egraph = EGraph()
bad_expr = egraph.let("expr", bad_expr)
egraph.run(containers_analysis_schedule)
bad_expr, cost = egraph.extract(bad_expr, include_cost=True, cost_model=param_cost_model)

print(cost.floats, cost.node_count)
print(render_num(bad_expr))

4 22
-0.03144312866911644 * (-3.0969157782045578 + -31.196859437348742 * (-0.044758903858526766 * x0 + x0 * x0 * exp(x0 * x0) ** (-1.0)) - x1)


In [ ]:
import time

from egglog.exp.param_eq.pipeline import _new_rewrite_scheduler

init = time.perf_counter()

expr = bad_expr
for _ in range(2):
    egraph = EGraph()
    expr = egraph.let("expr", expr)
    egraph.run(
        ((binary_analysis_schedule + run(binary_rewrite_ruleset, scheduler=_new_rewrite_scheduler())) * 30)
        + binary_analysis_schedule
    )
    expr, cost = egraph.extract(expr, include_cost=True, cost_model=param_cost_model)
print("total time", time.perf_counter() - init)

print(cost.floats, cost.node_count)
print(render_num(expr))


total time 0.08043487498071045
4 20
-0.03144312866911644 * (-3.0969157782045578 + -31.196859437348742 * (x0 * (-0.044758903858526766 + exp(x0 * x0) ** (-1.0) * x0)) - x1)


We see this only take 0.0 seconds!

In [6]:
init = time.perf_counter()

expr = bad_expr_container
reports = []
for _ in range(2):
    egraph = EGraph()
    expr = egraph.let("expr", expr)
    start = time.perf_counter()
    reports.append(
        egraph.run(
            ((containers_analysis_schedule + run(container_rewrite_ruleset, scheduler=_new_rewrite_scheduler())) * 30)
            + containers_analysis_schedule
        )
    )
    print("run time", time.perf_counter() - start)

    expr, cost = egraph.extract(expr, include_cost=True, cost_model=container_cost_model)

print("total time", time.perf_counter() - init)
print(cost.floats, cost.node_count)
print(render_num(expr))


run time 4.426236416969914
run time 0.023161708028055727
total time 4.800521750003099
4 18
-0.03144312866911644 * (-3.0969157782045578 - 31.196859437348742 * (x0 * (-0.044758903858526766 + x0 / exp(x0 ** 2.0))) - x1)


This takes four seconds! And most of it in the first run

In [7]:
import datetime

for report in reports:
    all_times = [
        *report.rebuild_time_per_ruleset.values(),
        *report.merge_time_per_ruleset.values(),
        *report.search_and_apply_time_per_ruleset.values(),
    ]
    print(sum(all_times, start=datetime.timedelta()))

0:00:00.004352
0:00:00.000012


But why does the run report list less than a second?